In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

LLM_MODEL_NAME="jaro-jm/llm:latest"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=2500, num_predict=20000)

text = "Za co Trump uważał wojnę w Ukrainie?"

In [ ]:
qgen_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an advanced Query Expansion Assistant for a vector database (Qdrant) search engine.
Your goal is to improve retrieval accuracy by generating related search queries based on the user's input.

Instructions:
1. Analyze the user's question (which will be in Polish).
2. Generate a list of 3-5 distinct, semantically rich search queries in Polish.
3. These queries should:
   - Use specific domain terminology, synonyms, and related entities.
   - Break down broad concepts into specific sub-topics.
   - Target different angles of the topic to maximize vector space coverage.
4. Output ONLY the list of questions, one per line. Do not include introductory text or numbering."""),

    ("user", "Jakie są relacje między Chinami a Rosją w kontekście wojny?"),
    ("ai", """Czy Chiny udzielają Rosji bezpośredniej pomocy wojskowej w konflikcie na Ukrainie?
Jak zmienił się wolumen handlu surowcami energetycznymi między Rosją, a Chinami po nałożeniu sankcji?
Jakie jest oficjalne stanowisko Pekinu wobec referendów aneksyjnych na okupowanych terytoriach?
W jaki sposób Chiny wykorzystują osłabienie Rosji do zwiększenia wpływów w Azji Centralnej?
Czy istnieją dowody na dostarczanie przez chińskie firmy technologii podwójnego zastosowania dla rosyjskiej armii?"""),

    ("user", "{text}")
])

qgen_chain = qgen_prompt | llm_text
questions_raw = qgen_chain.invoke({"text": text})

In [ ]:
content = questions_raw.content
questions_list = [q.strip() for q in content.splitlines() if q.strip()]

print("### Generated helper questions:\n")
for i, question in enumerate(questions_list, 1):
    print(f"{i}. {question}")